# Merge Result-Property Descriptions into the Object-Types RST Reference

Pure text processing, no Toolkit calls needed - merges `result_property_descriptions.json` (from `result_props_descriptions.ipynb`) into `object_types_props_results_snippet_global_mapped.rst` (from `object_types.ipynb`), widening every "Result Properties" table from one column (Name) to three (Name, Description, Standard Physical Unit). Writes a new file here in `output/` - does not touch `docs/source/object_types_props_results_snippet_global_mapped.rst` itself; that gets copied over manually once reviewed.

## Load inputs

In [1]:
import re
import json
from pathlib import Path

output_dir = Path.cwd().resolve() / "output"
rst_path = output_dir / "object_types_props_results_snippet_global_mapped.rst"
desc_path = output_dir / "result_property_descriptions.json"

rst_text = rst_path.read_text(encoding="utf-8")
desc_map = json.loads(desc_path.read_text(encoding="utf-8"))
len(rst_text), len(desc_map)

(141640, 776)

## Widen every "Result Properties" table

Object-type headings in this (mapped) RST are the table codes (e.g. `ROHR`), matching the `OBJTYPE` half of `result_property_descriptions.json`'s `OBJTYPE~ATTRTYPE` keys directly - no further mapping needed. Sections with "No result properties found." are left untouched. Missing lookups (e.g. `MAINELEMENT`, a toolkit-reported result property with no corresponding `.MX1` channel in the source data) get an empty Description/Unit cell rather than being dropped, so the property itself is never lost from the reference. Description/Unit are wrapped in double backticks like the existing Name/Value-Type columns - also neutralizes stray `*`/`|` characters some units contain (e.g. `[KW|MW]`), which would otherwise be parsed as RST markup.

In [2]:
heading_pat = re.compile(r"^(?P<name>[^\r\n]+)\r?\n\^{3,}\r?\n", re.MULTILINE)
matches = list(heading_pat.finditer(rst_text))

result_block_pat = re.compile(
    r"(Result Properties\r?\n\"+\r?\n\r?\n)"
    r"(\.\. list-table::\r?\n\s*:header-rows: 1\r?\n\r?\n\s*\* - Name\r?\n)"
    r"((?:\s*\* - ``[^`]+``\r?\n)+)"
)
row_pat = re.compile(r"\* - ``([^`]+)``")

def build_new_header(header_block):
    return header_block.rstrip("\n") + "\n     - Description\n     - Standard Physical Unit\n"

def build_new_rows(rows_text, section_name):
    out = []
    for p in row_pat.findall(rows_text):
        entry = desc_map.get(f"{section_name}~{p}")
        # .strip(): a stray leading/trailing space right against the closing ``
        # breaks RST inline-literal parsing (seen once in the source MX1 TITLE data itself).
        title = (entry["TITLE"] or "").strip() if entry else ""
        unit = (entry["UNIT"] or "").strip() if entry else ""
        title_cell = f"``{title}``" if title else ""
        unit_cell = f"``{unit}``" if unit else ""
        out.append(f"   * - ``{p}``\n     - {title_cell}\n     - {unit_cell}\n")
    return "".join(out)

In [3]:
replacements = []
sections_matched = 0
rows_total = 0
rows_described = 0

for i, m in enumerate(matches):
    start = m.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(rst_text)
    section_name = m.group("name").strip()
    section_text = rst_text[start:end]

    rb = result_block_pat.search(section_text)
    if not rb:
        continue
    sections_matched += 1

    props = row_pat.findall(rb.group(3))
    rows_total += len(props)
    rows_described += sum(1 for p in props if f"{section_name}~{p}" in desc_map)

    new_block = rb.group(1) + build_new_header(rb.group(2)) + build_new_rows(rb.group(3), section_name)
    replacements.append((start + rb.start(), start + rb.end(), new_block))

print(f"{sections_matched} object-type sections widened, "
      f"{rows_described}/{rows_total} result-property rows matched a description ({rows_described / rows_total:.1%})")

59 object-type sections widened, 760/840 result-property rows matched a description (90.5%)


In [4]:
parts = []
cursor = 0
for start, end, new_block in replacements:
    parts.append(rst_text[cursor:start])
    parts.append(new_block)
    cursor = end
parts.append(rst_text[cursor:])
new_rst_text = "".join(parts)
len(new_rst_text)

184542

## Fix up the preamble note

The source note (inherited from `object_types.ipynb`'s raw output) still says "Result properties are listed by name only." - no longer true now that Description/Unit are added. Replaced with an accurate sentence, and a note that Description/Unit are in German (as embedded in SIR 3S's own `.MX1` output).

In [5]:
OLD_NOTE_TAIL = "Result properties are listed by name only."
NEW_NOTE_TAIL = (
    "Result properties additionally list a Description and Standard Physical Unit where known, "
    "sourced from SIR 3S's own .MX1 output plus a handful of manual additions (see "
    "result_props_descriptions.ipynb). Both are in German, as embedded in SIR 3S itself."
)

assert OLD_NOTE_TAIL in new_rst_text, "Expected preamble sentence not found - has the source note changed?"
new_rst_text = new_rst_text.replace(OLD_NOTE_TAIL, NEW_NOTE_TAIL, 1)
print("Preamble note updated.")

Preamble note updated.


## Save

In [6]:
out_path = output_dir / "object_types_props_results_snippet_global_mapped_with_descriptions.rst"
out_path.write_text(new_rst_text, encoding="utf-8")
print(f"Wrote {out_path}")

Wrote C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_types_props_results_snippet_global_mapped_with_descriptions.rst
